# Pipeline maestro TumiPay

Este notebook queda como indice del flujo. La ejecucion real esta separada para evitar mezclar carga base, Power BI y RAG + LLM.

1. `notebooks/00_carga_base_supabase.ipynb`: carga limpia de `data/raw` a las tablas `raw_*` en Supabase.
2. `notebooks/01_powerbi_predicciones.ipynb`: genera y carga `predicciones_riesgo` para Power BI con UPSERT.
3. `notebooks/02_aporte_adicional_rag_llm.ipynb`: aporte adicional RAG + LLM para consultas en lenguaje natural.

In [2]:
import sys
import os
import importlib
from pathlib import Path

current_dir = Path(os.getcwd())
ROOT_DIR = current_dir if (current_dir / 'src').exists() else current_dir.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import src.config as _cfg_module
importlib.reload(_cfg_module)
from src.config import settings
from src.db_utils import create_supabase_engine, get_database_host

engine = create_supabase_engine(settings.DATABASE_URL)
print(f'Base de datos activa: {get_database_host(settings.DATABASE_URL)}')
print('La conexion local/Docker esta bloqueada para cargas de datos.')

ModuleNotFoundError: No module named 'pydantic_settings'

## Ejecucion por scripts

Equivalentes desde terminal:

- `.venv/bin/python scripts/load_raw_supabase.py`
- `.venv/bin/python scripts/load_powerbi_predictions.py`
- `.venv/bin/python scripts/populate_rag.py`

## Calidad de datos: Outlier de ingreso → Vista `v_clientes_bi`

Crea la vista en Supabase que capea `ingreso_mensual_estimado` al P99 dinámico para Power BI.
El registro con ~120.000.000 en estrato 2 queda marcado como `error_captura`; raw no se toca.

In [ ]:
from sqlalchemy import text

sql_view = """
CREATE OR REPLACE VIEW v_clientes_bi AS
WITH p AS (
    SELECT percentile_cont(0.99) WITHIN GROUP (ORDER BY ingreso_mensual_estimado) AS p99
    FROM raw_clientes
    WHERE ingreso_mensual_estimado IS NOT NULL
)
SELECT
    c.*,
    LEAST(c.ingreso_mensual_estimado, (SELECT p99 FROM p)) AS ingreso_mensual_bi,
    CASE
        WHEN c.ingreso_mensual_estimado > (SELECT p99 FROM p) AND COALESCE(c.estrato, 4) <= 3
            THEN 'error_captura'
        WHEN c.ingreso_mensual_estimado > (SELECT p99 FROM p)
            THEN 'outlier_real'
        ELSE 'normal'
    END AS flag_ingreso
FROM raw_clientes c;
"""

with engine.begin() as conn:
    conn.execute(text(sql_view))

print("✓ Vista v_clientes_bi creada/actualizada en Supabase.")

# Verificar el outlier
import pandas as pd
df_check = pd.read_sql("""
SELECT cliente_id, ingreso_mensual_estimado, ingreso_mensual_bi,
       flag_ingreso, estrato, ocupacion
FROM v_clientes_bi
WHERE flag_ingreso IN ('error_captura', 'outlier_real')
ORDER BY ingreso_mensual_estimado DESC;
""", engine)
display(df_check)

## Análisis de mora: proporción por rango de monto de crédito

¿Qué porcentaje del total de créditos está en mora, segmentado por tramos de monto?
Definición: `es_moroso = 1` si el crédito registró `max(dias_mora) > 30` en cuotas vencidas hasta el corte.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

sql_mora = """
WITH mora_base AS (
    SELECT
        cr.credito_id,
        cr.monto_credito,
        COALESCE(MAX(CASE WHEN p.dias_mora > 30 THEN 1 ELSE 0 END), 0) AS es_moroso
    FROM raw_creditos cr
    LEFT JOIN raw_pagos p
        ON cr.credito_id = p.credito_id
        AND p.fecha_vencimiento <= '2026-04-30'
    GROUP BY cr.credito_id, cr.monto_credito
),
rangos AS (
    SELECT
        credito_id,
        monto_credito,
        es_moroso,
        CASE
            WHEN monto_credito <  500000   THEN '1. < 500K'
            WHEN monto_credito <  1000000  THEN '2. 500K – 1M'
            WHEN monto_credito <  2000000  THEN '3. 1M – 2M'
            WHEN monto_credito <  5000000  THEN '4. 2M – 5M'
            WHEN monto_credito <  10000000 THEN '5. 5M – 10M'
            ELSE                                '6. > 10M'
        END AS rango_monto
    FROM mora_base
)
SELECT
    rango_monto,
    COUNT(*)                                                AS total_creditos,
    SUM(es_moroso)                                          AS creditos_mora,
    ROUND(SUM(es_moroso)::NUMERIC / COUNT(*) * 100, 2)     AS pct_mora,
    ROUND(SUM(monto_credito) / 1e6, 1)                     AS monto_total_MM,
    ROUND(SUM(CASE WHEN es_moroso = 1 THEN monto_credito ELSE 0 END) / 1e6, 1)  AS monto_mora_MM
FROM rangos
GROUP BY rango_monto
ORDER BY rango_monto;
"""

df_mora = pd.read_sql(sql_mora, engine)
df_mora['pct_mora'] = df_mora['pct_mora'].astype(float)
display(df_mora)

# ── Gráfico doble: barras apiladas (vol) + línea (% mora)
fig, ax1 = plt.subplots(figsize=(11, 5))
ax2 = ax1.twinx()

x = range(len(df_mora))
bar_total = ax1.bar(x, df_mora['total_creditos'], color='#4C72B0', alpha=0.65, label='Créditos sin mora')
bar_mora  = ax1.bar(x, df_mora['creditos_mora'],  color='#DD4444', alpha=0.90, label='En mora')

ax2.plot(x, df_mora['pct_mora'], color='#E67E22', marker='o', linewidth=2.2, label='% Mora')
for i, v in enumerate(df_mora['pct_mora']):
    ax2.annotate(f'{v:.1f}%', (i, v), textcoords='offset points', xytext=(0, 8),
                 ha='center', fontsize=9, color='#E67E22', fontweight='bold')

ax1.set_xticks(list(x))
ax1.set_xticklabels(df_mora['rango_monto'], rotation=15, ha='right')
ax1.set_ylabel('Número de créditos')
ax2.set_ylabel('% en mora sobre total del tramo')
ax2.set_ylim(0, df_mora['pct_mora'].max() * 1.5)
ax1.set_title('Proporción de créditos en mora por rango de monto', fontsize=13, fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{int(v):,}'))
plt.tight_layout()
plt.savefig('data/processed/mora_por_rango_monto.png', dpi=150)
plt.show()
print("Gráfico guardado en data/processed/mora_por_rango_monto.png")